# National Electricity Grid Network Analysis
Synthetic dataset generator + exploratory analysis for Ghana's grid (ECG, NEDCo, GRIDCo, VRA) with WAPP cross-border interconnections.

Produces three CSVs in the same spirit as the OpenFlights airlines/airports/routes trio:
- `utilities.csv` (like airlines.csv)
- `substations.csv` (like airports.csv)
- `lines.csv` (like routes.csv)

Grounded in Ghana's grid with cross-border interconnections reflecting the real West African Power Pool (WAPP), so the network has genuine international structure like the aviation network did.

In [ ]:
import csv
import math
import random
import os
import pandas as pd
import matplotlib.pyplot as plt
import folium
from geopy.distance import geodesic

random.seed(42)

## 1. Define utilities

In [ ]:
utilities = [
    # utility_id, name, alias, code, type, country, active
    (1, "Electricity Company of Ghana", "ECG", "ECG", "Distribution", "Ghana", "Y"),
    (2, "Northern Electricity Distribution Company", "NEDCo", "NED", "Distribution", "Ghana", "Y"),
    (3, "Ghana Grid Company", "GRIDCo", "GRD", "Transmission", "Ghana", "Y"),
    (4, "Volta River Authority", "VRA", "VRA", "Generation", "Ghana", "Y"),
    (5, "Compagnie Ivoirienne d'Electricite", "CIE", "CIE", "Distribution", "Cote d'Ivoire", "Y"),
    (6, "Communaute Electrique du Benin", "CEB", "CEB", "Transmission", "Togo/Benin", "Y"),
    (7, "Societe Beninoise d'Energie Electrique", "SBEE", "SBE", "Distribution", "Benin", "Y"),
    (8, "Electricite de Guinee", "EDG", "EDG", "Distribution", "Guinea", "N"),
    (9, "Sonabel", "SONABEL", "SNB", "Distribution", "Burkina Faso", "Y"),
    (10, "Enclave Power Company", "EPC", "EPC", "Generation", "Ghana", "N"),
]

## 2. Define substations (Ghana regions + cross-border)

In [ ]:
# ---------------------------------------------------------------------------
# Substations (analogous to airports.csv)
# Coordinates are approximate/illustrative, not survey-grade.
# ---------------------------------------------------------------------------
ghana_regions = {
    "Greater Accra": [
        ("Achimota", 5.614, -0.224), ("Tema", 5.669, -0.017), ("Mallam", 5.560, -0.298),
        ("Legon", 5.650, -0.186), ("Kaneshie", 5.560, -0.238), ("Aboadze Junction", 5.590, -0.150),
    ],
    "Ashanti": [
        ("Kumasi Central", 6.688, -1.624), ("Ejisu", 6.719, -1.463), ("Obuasi", 6.202, -1.663),
        ("Mampong", 7.062, -1.400), ("Konongo", 6.618, -1.219),
    ],
    "Western": [
        ("Takoradi", 4.895, -1.759), ("Aboadze", 4.985, -1.766), ("Tarkwa", 5.301, -1.994),
        ("Axim", 4.867, -2.241),
    ],
    "Central": [
        ("Cape Coast", 5.106, -1.246), ("Winneba", 5.352, -0.622), ("Kasoa", 5.533, -0.416),
        ("Assin Fosu", 5.699, -1.492),
    ],
    "Eastern": [
        ("Koforidua", 6.094, -0.259), ("Akosombo", 6.300, 0.055), ("Nkawkaw", 6.550, -0.767),
        ("Suhum", 6.041, -0.451),
    ],
    "Volta": [
        ("Ho", 6.611, 0.471), ("Kpong", 6.150, 0.100), ("Hohoe", 7.152, 0.472),
        ("Sogakope", 6.007, 0.573),
    ],
    "Bono": [
        ("Sunyani", 7.339, -2.326), ("Techiman", 7.590, -1.938), ("Berekum", 7.453, -2.585),
    ],
    "Northern": [
        ("Tamale", 9.403, -0.842), ("Yendi", 9.442, -0.011), ("Savelugu", 9.625, -0.826),
    ],
    "Upper East": [
        ("Bolgatanga", 10.787, -0.851), ("Bawku", 11.058, -0.243),
    ],
    "Upper West": [
        ("Wa", 10.061, -2.501),
    ],
}

cross_border = [
    # (name, country, lat, lon)
    ("Bolgatanga Interconnection", "Burkina Faso border", 11.20, -0.75),
    ("Elubo Border Station", "Cote d'Ivoire border", 5.20, -2.85),
    ("Aflao Border Station", "Togo border", 6.12, 1.19),
    ("Lome Transmission Hub", "Togo", 6.13, 1.22),
    ("Cotonou Transmission Hub", "Benin", 6.37, 2.43),
    ("Abidjan Transmission Hub", "Cote d'Ivoire", 5.35, -4.00),
    ("Bobo-Dioulasso Hub", "Burkina Faso", 11.18, -4.30),
    ("Conakry Transmission Hub", "Guinea", 9.64, -13.58),
]

voltage_levels = [11, 33, 69, 161, 330]
sub_types = ["Distribution", "Bulk Supply Point", "Transmission"]

In [ ]:
substations = []
sid = 1
name_to_id = {}
for region, places in ghana_regions.items():
    for name, lat, lon in places:
        voltage = random.choice(voltage_levels)
        sub_type = "Transmission" if voltage >= 161 else ("Bulk Supply Point" if voltage == 69 else "Distribution")
        capacity = round(random.uniform(15, 400) if sub_type != "Distribution" else random.uniform(5, 60), 1)
        status = "Active" if random.random() > 0.05 else "Inactive"
        substations.append([
            sid, f"{name} Substation", name, region, "Ghana",
            round(lat + random.uniform(-0.01, 0.01), 4), round(lon + random.uniform(-0.01, 0.01), 4),
            voltage, capacity, random.randint(1965, 2023), sub_type, status,
        ])
        name_to_id[name] = sid
        sid += 1

for name, country, lat, lon in cross_border:
    voltage = random.choice([161, 330])
    capacity = round(random.uniform(100, 500), 1)
    substations.append([
        sid, f"{name}", name, country, country.split()[0],
        round(lat, 4), round(lon, 4), voltage, capacity,
        random.randint(1980, 2020), "Transmission", "Active",
    ])
    name_to_id[name] = sid
    sid += 1

## 3. Build transmission/distribution lines

In [ ]:
# ---------------------------------------------------------------------------
# Transmission/Distribution lines (analogous to routes.csv)
# ---------------------------------------------------------------------------
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))

sub_by_id = {row[0]: row for row in substations}
sub_ids = list(sub_by_id.keys())

In [ ]:
lines = []
lid = 1
seen_pairs = set()

# Connect substations within each region into a loosely meshed network
for region, places in ghana_regions.items():
    ids_in_region = [name_to_id[n] for n, _, _ in places]
    for i, a in enumerate(ids_in_region):
        for b in ids_in_region[i + 1:]:
            if random.random() < 0.55:
                pair = tuple(sorted((a, b)))
                if pair in seen_pairs:
                    continue
                seen_pairs.add(pair)
                utility_id = random.choice([1, 2, 3])
                sub_a, sub_b = sub_by_id[a], sub_by_id[b]
                dist = round(haversine_km(sub_a[5], sub_a[6], sub_b[5], sub_b[6]) * random.uniform(1.05, 1.3), 1)
                voltage = min(sub_a[7], sub_b[7])
                lines.append([
                    lid, utility_id, a, sub_a[1], b, sub_b[1],
                    voltage, dist, round(random.uniform(20, 300), 1),
                    "Active" if random.random() > 0.08 else "Under Maintenance",
                    random.choice(["Overhead", "Underground"]),
                ])
                lid += 1

In [ ]:
# A handful of inter-regional backbone lines (transmission-level, GRIDCo)
region_hub = {r: name_to_id[places[0][0]] for r, places in ghana_regions.items()}
region_names = list(region_hub.keys())
for i in range(len(region_names) - 1):
    a = region_hub[region_names[i]]
    b = region_hub[region_names[i + 1]]
    sub_a, sub_b = sub_by_id[a], sub_by_id[b]
    dist = round(haversine_km(sub_a[5], sub_a[6], sub_b[5], sub_b[6]) * 1.15, 1)
    lines.append([
        lid, 3, a, sub_a[1], b, sub_b[1], 330, dist,
        round(random.uniform(200, 600), 1), "Active", "Overhead",
    ])
    lid += 1

In [ ]:
# Cross-border interconnections (WAPP-style)
border_links = [
    ("Bolgatanga", "Bolgatanga Interconnection", 9),
    ("Bolgatanga Interconnection", "Bobo-Dioulasso Hub", 9),
    ("Elubo Border Station", "Kumasi Central", 5),
    ("Elubo Border Station", "Abidjan Transmission Hub", 5),
    ("Aflao Border Station", "Tema", 6),
    ("Aflao Border Station", "Lome Transmission Hub", 6),
    ("Lome Transmission Hub", "Cotonou Transmission Hub", 6),
]
for src_name, dst_name, utility_id in border_links:
    if src_name not in name_to_id or dst_name not in name_to_id:
        continue
    a, b = name_to_id[src_name], name_to_id[dst_name]
    sub_a, sub_b = sub_by_id[a], sub_by_id[b]
    dist = round(haversine_km(sub_a[5], sub_a[6], sub_b[5], sub_b[6]) * 1.1, 1)
    lines.append([
        lid, utility_id, a, sub_a[1], b, sub_b[1], 330, dist,
        round(random.uniform(150, 400), 1), "Active", "Overhead",
    ])
    lid += 1

## 4. Write CSVs

In [ ]:
with open("utilities.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Utility ID", "Name", "Alias", "Code", "Type", "Country", "Active"])
    w.writerows(utilities)

with open("substations.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Substation ID", "Name", "Short Name", "Region", "Country", "Latitude", "Longitude",
                "Voltage (kV)", "Capacity (MVA)", "Commissioning Year", "Type", "Status"])
    w.writerows(substations)

with open("lines.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Line ID", "Utility ID", "Source Substation ID", "Source Substation",
                "Destination Substation ID", "Destination Substation", "Voltage (kV)",
                "Length (km)", "Capacity (MVA)", "Status", "Line Type"])
    w.writerows(lines)

print(f"utilities: {len(utilities)} rows")
print(f"substations: {len(substations)} rows")
print(f"lines: {len(lines)} rows")

## 5. Reload and validate coordinates

In [ ]:
utilities = pd.read_csv("utilities.csv")
substations = pd.read_csv("substations.csv")
lines = pd.read_csv("lines.csv")

# Now inspect the location data
print(substations[["Name", "Latitude", "Longitude"]])

# Now validate the coordinates
invalid = substations[
    (substations["Latitude"] < 4) |
    (substations["Latitude"] > 11) |
    (substations["Longitude"] < -3) |
    (substations["Longitude"] > 1)
]

print(invalid)

## 6. Interactive map — substations by voltage tier

In [ ]:
substations = pd.read_csv("substations.csv")

# Create a map centered roughly on Ghana
ghana_map = folium.Map(location=[7.95, -1.02], zoom_start=7)

# Add a marker for each substation
# Color map by voltage
voltage_colors = {
    11: 'lightblue', 33: 'blue', 69: 'green',
    161: 'orange', 330: 'red'
}

# Create a FeatureGroup per voltage tier
layers = {v: folium.FeatureGroup(name=f"{v} kV") for v in voltage_colors}

for _, row in substations.iterrows():
    voltage = row["Voltage (kV)"]
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=5,
        popup=f"{row['Name']} ({voltage} kV)",
        color=voltage_colors.get(voltage, 'gray'),
        fill=True,
        fill_opacity=0.8
    ).add_to(layers[voltage])

# Add all layers to the map, then the toggle control
for layer in layers.values():
    layer.add_to(ghana_map)

folium.LayerControl().add_to(ghana_map)

# Save the map
ghana_map.save("substations_map.html")

print(os.path.abspath("substations_map.html"))

ghana_map

## 7. Full analysis — regional connectivity, distances, gaps, utility territories

In [ ]:
# ── Load data ──────────────────────────────────────────────
substations = pd.read_csv('substations.csv')
lines = pd.read_csv('lines.csv')
utilities = pd.read_csv('utilities.csv')

# ── Merge lines with source + destination substation info ──
lines_with_region = lines.merge(
    substations[['Substation ID', 'Name', 'Region', 'Country']],
    left_on='Source Substation ID', right_on='Substation ID',
    how='left', suffixes=('', '_source')
)
lines_with_region = lines_with_region.merge(
    substations[['Substation ID', 'Name', 'Region', 'Country']],
    left_on='Destination Substation ID', right_on='Substation ID',
    how='left', suffixes=('_source', '_dest')
)
lines_with_utility = lines_with_region.merge(
    utilities[['Utility ID', 'Name', 'Code']], on='Utility ID', how='left'
)

### 7a. Regional connectivity

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. REGIONAL CONNECTIVITY
# ═══════════════════════════════════════════════════════════
substations['Region'].value_counts().plot(kind='bar', title='Substations by Region')
plt.ylabel('Count'); plt.tight_layout(); plt.savefig('regional_substations.png'); plt.show()

lines_with_region.groupby('Region_source').size().plot(kind='bar', title='Lines by Source Region')
plt.ylabel('Count'); plt.tight_layout(); plt.savefig('regional_lines.png'); plt.show()

border_regions = ["Burkina Faso", "Cote d'Ivoire", "Togo", "Benin", "Guinea",
                   "Burkina Faso border", "Cote d'Ivoire border", "Togo border"]
substations['is_border'] = substations['Region'].isin(border_regions)
substations['is_border'].value_counts().plot(kind='bar', title='Domestic vs Cross-Border Substations')
plt.tight_layout(); plt.savefig('border_vs_domestic.png'); plt.show()

print("Top region by substation count:", substations['Region'].value_counts().idxmax())

### 7b. Distance analysis

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. DISTANCE ANALYSIS
# ═══════════════════════════════════════════════════════════
bins = [0, 20, 60, lines['Length (km)'].max()]
labels = ['Short (<20km)', 'Medium (20-60km)', 'Long (>60km)']
lines['Distance Category'] = pd.cut(lines['Length (km)'], bins=bins, labels=labels)

lines['Distance Category'].value_counts().reindex(labels).plot(
    kind='bar', title='Line Length Distribution')
plt.ylabel('Number of Lines'); plt.tight_layout(); plt.savefig('distance_distribution.png'); plt.show()

print(lines.groupby('Distance Category')['Length (km)'].describe())

### 7c. Geographic gaps

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. GEOGRAPHIC GAPS
# ═══════════════════════════════════════════════════════════
region_counts = substations[~substations['is_border']]['Region'].value_counts()
underserved = region_counts.sort_values().head(3)
print("Regions with the fewest substations (potentially underserved):")
print(underserved)

region_counts.sort_values().plot(kind='barh', title='Substation Count by Region (Ascending)')
plt.tight_layout(); plt.savefig('geographic_gaps.png'); plt.show()

### 7d. Utility territory map

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. UTILITY TERRITORY MAP
# ═══════════════════════════════════════════════════════════
territory_map = folium.Map(location=[7.95, -1.02], zoom_start=7)
utility_colors = ['red', 'blue', 'green', 'purple', 'orange',
                   'darkred', 'cadetblue', 'darkgreen', 'black', 'pink']

sub_lookup = substations.set_index('Substation ID')

for i, (_, util) in enumerate(utilities.iterrows()):
    color = utility_colors[i % len(utility_colors)]
    fg = folium.FeatureGroup(name=util['Alias'])
    util_lines = lines[lines['Utility ID'] == util['Utility ID']]

    for _, ln in util_lines.iterrows():
        try:
            src = sub_lookup.loc[ln['Source Substation ID']]
            dst = sub_lookup.loc[ln['Destination Substation ID']]
            folium.PolyLine(
                locations=[[src['Latitude'], src['Longitude']],
                           [dst['Latitude'], dst['Longitude']]],
                color=color, weight=2,
                popup=f"{util['Alias']} line"
            ).add_to(fg)
        except KeyError:
            continue
    fg.add_to(territory_map)

folium.LayerControl().add_to(territory_map)
territory_map.save('utility_territory_map.html')
print("Utility territory map saved.")

territory_map

### 7e. High-capacity substation clustering

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. SUBSTATION CLUSTERING (high-capacity)
# ═══════════════════════════════════════════════════════════
threshold = substations['Capacity (MVA)'].quantile(0.75)
high_cap = substations[substations['Capacity (MVA)'] >= threshold]
print(f"Capacity threshold (75th percentile): {threshold:.1f} MVA")
print(f"{len(high_cap)} high-capacity substations found")
print(high_cap['Region'].value_counts())

cluster_map = folium.Map(location=[7.95, -1.02], zoom_start=7)
for _, sub in high_cap.iterrows():
    folium.CircleMarker(
        location=[sub['Latitude'], sub['Longitude']],
        radius=7, color='crimson', fill=True, fill_opacity=0.8,
        popup=f"{sub['Name']} ({sub['Capacity (MVA)']} MVA)"
    ).add_to(cluster_map)
cluster_map.save('high_capacity_clusters.html')
print("Substation clustering map saved.")

cluster_map